# 02 — Main workflow

Runs the full compression sweep on all 12 pretrained ViT-Base models:

1. Baseline accuracy on ImageNet-100 val (reference)
2. Magnitude pruning sweep (4 scopes × 7 ratios)
3. Structured pruning sweep (heads + neurons, 2 scopes)
4. PTQ sweep (INT8/4/2 across 3 scopes)
5. PE-buffer secondary experiments (magnitude + PTQ on PE params only)
6. CKA analysis on pruned models
7. Aggregate analysis (tables, CSVs)

All steps support `--skip_existing`-style resume by checking the output JSON
before running. If a Colab session disconnects, restart from this notebook.

---

## 🔧 Configuration required

Before running, you must set the following paths:

- **Cell 1.3** — `REPO_SRC` *(only if you cloned the repo to a custom Drive location; otherwise leave default)*
- **Cell 1.4** — `CHECKPOINT_ROOT` and `RESULTS_ROOT` *(your Drive paths)*
- **Cell 1.5** — `IMAGENET_TAR` *(path to your ILSVRC2012 val tar)*

All other paths in the notebook are derived from these or use ephemeral `/content/` storage; do **not** change them.

## 1. Setup

In [ ]:
# 1.1 GPU check  (no changes needed)
import torch
print('CUDA:', torch.cuda.is_available(), '|',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# 1.2 Mount Drive  (no changes needed)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 1.3 Bring the repo into /content
# ===========================================================================
# >>> USER CONFIGURATION (optional) <<<
# Set REPO_SRC to the folder on YOUR Google Drive where you unpacked
# this repository's code. The default below matches the layout used
# in the workflow documentation.
# ===========================================================================
import shutil, os
REPO_SRC = '/content/drive/MyDrive/pe_compression_experiment/code'  # <-- EDIT if needed

# ---------------------------------------------------------------------------
# Destination on the Colab instance  (no changes needed)
# ---------------------------------------------------------------------------
REPO_DST = '/content/vit-pe-compression'
if os.path.exists(REPO_DST):
    shutil.rmtree(REPO_DST)
shutil.copytree(REPO_SRC, REPO_DST)
%cd /content/vit-pe-compression

In [ ]:
# 1.4 Path configuration
# ===========================================================================
# >>> USER CONFIGURATION REQUIRED <<<
#
# Set the TWO paths below to match YOUR Google Drive layout:
#
#   CHECKPOINT_ROOT — folder holding the 12 pretrained ViT-Base checkpoints.
#                     Expected layout:
#                         <CHECKPOINT_ROOT>/learned_seed42/best_model.pth
#                         <CHECKPOINT_ROOT>/learned_seed123/best_model.pth
#                         ... (12 total)
#                     Download the checkpoints from the link in the repo README.
#
#   RESULTS_ROOT    — folder where this notebook will write all experiment
#                     outputs (baseline/, pruning/, quantization/, cka/,
#                     analysis/). Should live on YOUR Drive so artifacts
#                     persist between sessions. Will be created if needed.
# ===========================================================================
CHECKPOINT_ROOT = '/content/drive/MyDrive/Trained models_ImageNet100'                   # <-- EDIT THIS
RESULTS_ROOT    = '/content/drive/MyDrive/pe_compression_experiment/results'            # <-- EDIT THIS

# ---------------------------------------------------------------------------
# Derived paths and ephemeral cache  (no changes needed below this line)
# ---------------------------------------------------------------------------
VAL_ROOT        = '/content/imagenet100/val'    # ephemeral; created by Cell 1.5

BASELINE_JSON   = f'{RESULTS_ROOT}/baseline/baseline_accuracy.json'
MAGNITUDE_JSON  = f'{RESULTS_ROOT}/pruning/magnitude.json'
STRUCTURED_JSON = f'{RESULTS_ROOT}/pruning/structured.json'
PTQ_JSON        = f'{RESULTS_ROOT}/quantization/ptq.json'
ANALYSIS_DIR    = f'{RESULTS_ROOT}/analysis'

for d in [BASELINE_JSON, MAGNITUDE_JSON, STRUCTURED_JSON, PTQ_JSON]:
    os.makedirs(os.path.dirname(d), exist_ok=True)
os.makedirs(ANALYSIS_DIR, exist_ok=True)

print(f'REPO_DST:         {REPO_DST}')
print(f'CHECKPOINT_ROOT:  {CHECKPOINT_ROOT}')
print(f'RESULTS_ROOT:     {RESULTS_ROOT}')
print(f'VAL_ROOT:         {VAL_ROOT}')
print('Output paths ready.')

In [ ]:
# 1.5 Set up ImageNet-100 val if not already present
# ===========================================================================
# >>> USER CONFIGURATION REQUIRED <<<
# Set IMAGENET_TAR to the location of the official ILSVRC2012 validation
# tar archive on YOUR Drive. The archive must be downloaded separately from
# https://image-net.org/ (free academic registration required).
# ===========================================================================
IMAGENET_TAR = '/content/drive/MyDrive/pe_experiment/imagenet/ILSVRC2012_img_val.tar'  # <-- EDIT THIS

# ---------------------------------------------------------------------------
# Extraction target (ephemeral /content/ — recreated per session)  (no changes needed)
# ---------------------------------------------------------------------------
if not os.path.exists(VAL_ROOT):
    !python -m scripts.setup_imagenet100_val \
        --tar_path "$IMAGENET_TAR" \
        --output_dir /content/imagenet100
else:
    print(f'Val set already exists at {VAL_ROOT}')

## 2. Baseline evaluation

Top-1 accuracy of each pretrained model on ImageNet-100 val. 

In [ ]:
# (no changes needed)
!python -m scripts.baseline_eval \
    --checkpoint_root "$CHECKPOINT_ROOT" \
    --val_root $VAL_ROOT \
    --output_path $BASELINE_JSON \
    --batch_size 128

## 3. Magnitude pruning sweep

12 models × 4 scopes × 7 ratios = 336 evaluations. 

In [ ]:
# (no changes needed)
!python -m scripts.magnitude_pruning \
    --checkpoint_root "$CHECKPOINT_ROOT" \
    --val_root $VAL_ROOT \
    --output_path $MAGNITUDE_JSON \
    --batch_size 128

## 4. Structured pruning sweep

12 models × 2 scopes × (6 head ratios + 5 neuron ratios) ≈ 264 evaluations.

In [ ]:
# (no changes needed)
!python -m scripts.structured_pruning \
    --checkpoint_root "$CHECKPOINT_ROOT" \
    --val_root $VAL_ROOT \
    --output_path $STRUCTURED_JSON \
    --batch_size 128

## 5. Post-training quantization sweep

12 models × 3 scopes × 4 bit widths (32, 8, 4, 2) = 144 evaluations.

In [ ]:
# (no changes needed)
!python -m scripts.ptq_quantization \
    --checkpoint_root "$CHECKPOINT_ROOT" \
    --val_root $VAL_ROOT \
    --output_path $PTQ_JSON \
    --batch_size 128

## 6. PE-buffer secondary experiments

Targets the PE parameters/buffers themselves. Writes to separate JSON
outputs so the primary results stay clean.

In [ ]:
# 6.1 Magnitude pruning, PE buffer only  (no changes needed)
!python -m scripts.magnitude_pruning \
    --checkpoint_root "$CHECKPOINT_ROOT" \
    --val_root $VAL_ROOT \
    --output_path $RESULTS_ROOT/pruning/magnitude_pe_buffer.json \
    --scopes pe_buffer_cache \
    --batch_size 128

In [ ]:
# 6.2 PTQ, PE buffer only  (no changes needed)
!python -m scripts.ptq_quantization \
    --checkpoint_root "$CHECKPOINT_ROOT" \
    --val_root $VAL_ROOT \
    --output_path $RESULTS_ROOT/quantization/ptq_pe_buffer.json \
    --scopes pe_buffer_cache \
    --batch_size 128

## 7. CKA analysis on pruned models

Layer-wise CKA between original and pruned variants, used to localize where
compression damages representations.

In [ ]:
# (no changes needed)
!python -m scripts.cka_pruning \
    --checkpoint_root "$CHECKPOINT_ROOT" \
    --val_root $VAL_ROOT \
    --output_path $RESULTS_ROOT/cka/cka_pruning.json \
    --batch_size 128

## 8. Aggregate analysis

In [ ]:
# (no changes needed)
!python -m scripts.analyze_results \
    --baseline $BASELINE_JSON \
    --magnitude $MAGNITUDE_JSON \
    --structured $STRUCTURED_JSON \
    --ptq $PTQ_JSON \
    --output_dir $ANALYSIS_DIR

## 9. Generate paper-ready tables and figures

Once the JSON outputs from sections 2–7 are populated, this section
produces the LaTeX tables and PDF figures used in the paper. Two
scripts are invoked:

- `generate_results_assets.py` produces the four main-text tables
  (`tab_baseline`, `tab_heads_global`, `tab_ptq_buffer`,
  `tab_pe_buffer_pruning`) and three main-text figures
  (`fig_heads_pruning`, `fig_pe_buffer`, `fig_cka_depth`).
- `generate_appendix_assets.py` produces the fourteen appendix
  tables that report the full compression grid.

Both scripts read directly from the JSON outputs and write LaTeX/PDF
files that can be `\input`-ed into the paper source tree. Reruns
are idempotent.

In [ ]:
# 9.1 Asset output paths  (no changes needed)
# ---------------------------------------------------------------------------
# Asset folders live under RESULTS_ROOT so the regenerated tables and
# figures persist alongside the JSON outputs. The paper's main.tex
# expects assets in a flat directory; for that workflow, copy the
# contents of these folders into the paper directory.
# ---------------------------------------------------------------------------
RESULTS_ASSETS_DIR  = f'{RESULTS_ROOT}/results_assets'
APPENDIX_ASSETS_DIR = f'{RESULTS_ROOT}/appendix_assets'

os.makedirs(RESULTS_ASSETS_DIR, exist_ok=True)
os.makedirs(APPENDIX_ASSETS_DIR, exist_ok=True)

# The generators expect JSON inputs in a flat directory. We point at
# RESULTS_ROOT and rely on the experiment scripts above having written
# the JSONs into RESULTS_ROOT's subfolders (baseline/, pruning/, etc.).
# The generator scripts know how to find each one by basename.
JSON_DIR = RESULTS_ROOT  # the generators expect flat layout; staging in 9.2 handles this

print(f'Main-text assets   → {RESULTS_ASSETS_DIR}')
print(f'Appendix assets    → {APPENDIX_ASSETS_DIR}')
print(f'JSON inputs from   → {JSON_DIR}')

In [ ]:
# 9.2 Stage JSON inputs in a single directory for the generators
#     (no changes needed)
# ---------------------------------------------------------------------------
# The two asset-generation scripts read JSON files by basename from a
# single --json_dir. The experiment scripts above wrote results into
# subfolders (baseline/, pruning/, quantization/, cka/). We stage
# symlinks/copies in a single directory so the generators can find them.
# ---------------------------------------------------------------------------
import shutil

STAGING_DIR = f'{RESULTS_ROOT}/_assets_staging'
os.makedirs(STAGING_DIR, exist_ok=True)

json_sources = {
    'baseline_accuracy.json'   : BASELINE_JSON,
    'magnitude.json'           : MAGNITUDE_JSON,
    'magnitude_pe_buffer.json' : f'{RESULTS_ROOT}/pruning/magnitude_pe_buffer.json',
    'structured.json'          : STRUCTURED_JSON,
    'ptq.json'                 : PTQ_JSON,
    'ptq_pe_buffer.json'       : f'{RESULTS_ROOT}/quantization/ptq_pe_buffer.json',
    'cka_pruning.json'         : f'{RESULTS_ROOT}/cka/cka_pruning.json',
}

for basename, src in json_sources.items():
    if os.path.exists(src):
        shutil.copy(src, f'{STAGING_DIR}/{basename}')
        print(f'  ✓ {basename}')
    else:
        print(f'  ✗ MISSING: {basename}  (expected at {src})')

print(f'\nStaged at: {STAGING_DIR}')

In [ ]:
# 9.3 Generate main-text tables and figures  (no changes needed)
!python -m scripts.generate_results_assets \
    --json_dir $STAGING_DIR \
    --out_dir $RESULTS_ASSETS_DIR

In [ ]:
# 9.4 Generate appendix tables  (no changes needed)
!python -m scripts.generate_appendix_assets \
    --json_dir $STAGING_DIR \
    --out_dir $APPENDIX_ASSETS_DIR

In [ ]:
# 9.5 List generated assets  (no changes needed)
print('Main-text assets generated:')
for f in sorted(os.listdir(RESULTS_ASSETS_DIR)):
    size_kb = os.path.getsize(f'{RESULTS_ASSETS_DIR}/{f}') / 1024
    print(f'  {f:<40s} ({size_kb:.1f} KB)')

print(f'\nAppendix tables generated:')
for f in sorted(os.listdir(APPENDIX_ASSETS_DIR)):
    size_kb = os.path.getsize(f'{APPENDIX_ASSETS_DIR}/{f}') / 1024
    print(f'  {f:<40s} ({size_kb:.1f} KB)')

## 10. Quick preview (Python-side)

In [ ]:
# (no changes needed)
import json
for label, path in [('baseline', BASELINE_JSON),
                     ('magnitude', MAGNITUDE_JSON),
                     ('structured', STRUCTURED_JSON),
                     ('ptq', PTQ_JSON)]:
    import os
    if os.path.exists(path):
        with open(path) as f:
            d = json.load(f)
        n = sum(1 for k in d if not k.startswith('_'))
        print(f'{label:<12}{n:>4} entries')
    else:
        print(f'{label:<12}MISSING')